# 03 — Structured Entity Extraction (serverless)

Reads the raw VLM output from notebook `02` (default table `video_inferences`) and structures it into a queryable Delta table (default `video_events`) using Databricks AI Functions.

The structuring step calls `databricks-meta-llama-3-3-70b-instruct` through `ai_query` with a `responseFormat` JSON schema. Server-side schema enforcement means the result always parses, so no `json.loads` error handling is needed downstream.

**Compute:** serverless CPU (no GPU needed), or a SQL warehouse. The work runs on the Foundation Model API endpoint, so this notebook does no heavy lifting locally.

**Prerequisite:** the `databricks-meta-llama-3-3-70b-instruct` Foundation Model API endpoint is available in the workspace (set `LLM_ENDPOINT` to use another one).

Differences from the classic version (`3_Video_Batch_Inference_Ray_Core`):
* **Schema fix.** Every field is listed in `required`, and `"strict": true` sits on `json_schema` rather than inside `schema`. Without both, the model left out `location`, `time_range` and `confidence` on every row, and the demo query returned nothing.
* The demo query at the end reads the table from the widgets instead of a hard-coded name.

In [0]:
dbutils.widgets.text("CATALOG", "main", label="CATALOG")
dbutils.widgets.text("SCHEMA", "vlm_video", label="SCHEMA")  # same as notebook 00
dbutils.widgets.text("INPUT_TABLE", "video_inferences", label="INPUT_TABLE")
dbutils.widgets.text("OUTPUT_TABLE", "video_events", label="OUTPUT_TABLE")
dbutils.widgets.text("LLM_ENDPOINT", "databricks-meta-llama-3-3-70b-instruct", label="LLM_ENDPOINT")

CATALOG = dbutils.widgets.get("CATALOG")
SCHEMA = dbutils.widgets.get("SCHEMA")
INPUT_TABLE = dbutils.widgets.get("INPUT_TABLE")
OUTPUT_TABLE = dbutils.widgets.get("OUTPUT_TABLE")
LLM_ENDPOINT = dbutils.widgets.get("LLM_ENDPOINT")

input_fqn = f"{CATALOG}.{SCHEMA}.{INPUT_TABLE}"
output_fqn = f"{CATALOG}.{SCHEMA}.{OUTPUT_TABLE}"
print(f"{input_fqn} -> {output_fqn} via {LLM_ENDPOINT}")

## Run AI Functions over the raw VLM outputs

The JSON schema is defined inline below. To adapt this notebook for a different use case, change two things only:
1. The instruction string passed to `ai_query` (currently asks for shoplifting incident metadata).
2. The JSON schema body (currently `shoplifting_incident_metadata`).

Everything else (the table names, the LLM endpoint, the `from_json` parsing) is generic.

In [0]:
# Three improvements over the original schema:
#   1. assessment is a 5-level enum, not a binary boolean. Recovers VLM prose
#      that says "suspected" / "appears to be" / "likely" — these were being
#      classified as incident_confirmed=false in the binary version.
#   2. Every field is listed in "required" and "strict" sits on json_schema (not inside schema);
#      without both, the model silently omitted location/time_range/confidence on every row.
#   3. schema_of_json is replaced with explicit struct DDL so column types are
#      correct (BOOLEAN / DOUBLE / ARRAY<STRUCT<...>>) instead of all STRING.
RESULT_STRUCT = (
    "STRUCT<"
    "timestamp:STRING, "
    "time_range:STRING, "
    "location:STRING, "
    "camera_view:STRING, "
    "suspects:ARRAY<STRUCT<"
    "person_id:STRING, appearance:STRING, actions:ARRAY<STRING>, "
    "item_taken:STRING, concealment_method:STRING, exit_behavior:STRING"
    ">>, "
    "assessment:STRING, "
    "confidence:DOUBLE, "
    "witnesses_or_staff_present:BOOLEAN, "
    "response_observed:STRING, "
    "notable_behaviors:ARRAY<STRING>, "
    "video_quality_notes:STRING, "
    "contextual_notes:STRING"
    ">"
)

INSTRUCTION = (
    "You are a loss-prevention analyst classifying a CCTV-footage analysis. "
    "Read the description below and extract structured incident metadata. "
    "Use this scale for the assessment field: "
    "CONFIRMED = analyst explicitly states shoplifting occurred with clear evidence "
    "(e.g. concealment shown, exit without payment). "
    "LIKELY = analyst describes suspicious behaviour consistent with shoplifting but "
    "stops short of formal confirmation (uses words like suspected, appears to be, "
    "likely). "
    "POSSIBLE = a behaviour that could be suspicious but is also consistent with "
    "normal customer behaviour. "
    "NORMAL = analyst describes normal customer behaviour with no theft indicators. "
    "INSUFFICIENT_EVIDENCE = description too sparse, ambiguous, or video quality too "
    "poor to make any assessment. "
    "Confidence is 0.0-1.0 reflecting the analyst certainty in their description. "
    "Description: "
)

sql = f"""
CREATE OR REPLACE TABLE {output_fqn} AS
SELECT
  *,
  parsed_result.timestamp AS timestamp,
  parsed_result.time_range AS time_range,
  parsed_result.location AS location,
  parsed_result.camera_view AS camera_view,
  parsed_result.suspects AS suspects,
  parsed_result.assessment AS assessment,
  parsed_result.confidence AS confidence,
  parsed_result.witnesses_or_staff_present AS witnesses_or_staff_present,
  parsed_result.response_observed AS response_observed,
  parsed_result.notable_behaviors AS notable_behaviors,
  parsed_result.video_quality_notes AS video_quality_notes,
  parsed_result.contextual_notes AS contextual_notes
FROM (
  SELECT *,
    from_json(
      ai_query(
        '{LLM_ENDPOINT}',
        CONCAT('{INSTRUCTION}', generated_text),
        responseFormat => '{{
          "type": "json_schema",
          "json_schema": {{
            "name": "shoplifting_incident_metadata",
            "schema": {{
              "type": "object",
              "properties": {{
                "timestamp": {{ "type": "string", "description": "On-screen recording date/time if visible, else empty string" }},
                "time_range": {{ "type": "string", "description": "Time range in the video when the incident occurs (e.g., 03:12 - 03:48)" }},
                "location": {{ "type": "string", "description": "General store area (e.g., cosmetics aisle, electronics section)" }},
                "camera_view": {{ "type": "string", "description": "Camera perspective or label if visible (e.g., CAM3, front entrance)" }},
                "suspects": {{
                  "type": "array",
                  "items": {{
                    "type": "object",
                    "properties": {{
                      "person_id": {{ "type": "string", "description": "Identifier or label for tracking this individual (e.g., Person 1)" }},
                      "appearance": {{ "type": "string", "description": "Clothing and distinguishing features" }},
                      "actions": {{
                        "type": "array",
                        "items": {{ "type": "string" }},
                        "description": "Sequence of actions that indicate suspicious or shoplifting behavior"
                      }},
                      "item_taken": {{ "type": "string", "description": "Description of the item suspected to be stolen (if identifiable)" }},
                      "concealment_method": {{ "type": "string", "description": "How the item was hidden (e.g., in bag, under jacket)" }},
                      "exit_behavior": {{ "type": "string", "description": "How the suspect exited (e.g., ran out, walked past checkout)" }}
                    }},
                    "required": ["person_id", "appearance", "actions", "item_taken", "concealment_method", "exit_behavior"],
                    "additionalProperties": false
                  }}
                }},
                "assessment": {{
                  "type": "string",
                  "enum": ["CONFIRMED", "LIKELY", "POSSIBLE", "NORMAL", "INSUFFICIENT_EVIDENCE"],
                  "description": "Multi-class incident assessment level"
                }},
                "confidence": {{ "type": "number", "minimum": 0, "maximum": 1, "description": "Analyst certainty (0.0 to 1.0)" }},
                "witnesses_or_staff_present": {{ "type": "boolean", "description": "True if other people (e.g., staff or customers) witnessed the event" }},
                "response_observed": {{ "type": "string", "description": "Any staff/security reaction captured in the footage" }},
                "notable_behaviors": {{
                  "type": "array",
                  "items": {{ "type": "string" }},
                  "description": "Additional suspicious behaviors observed (e.g., pacing, watching staff)"
                }},
                "video_quality_notes": {{ "type": "string", "description": "Notes on visibility, resolution, obstructions, or clarity" }},
                "contextual_notes": {{ "type": "string", "description": "Other useful context (e.g., group coordination, distraction tactics, time of day)" }}
              }},
              "required": ["timestamp", "time_range", "location", "camera_view", "suspects", "assessment", "confidence", "witnesses_or_staff_present", "response_observed", "notable_behaviors", "video_quality_notes", "contextual_notes"],
              "additionalProperties": false
            }},
            "strict": true
          }}
        }}'
      ),
      '{RESULT_STRUCT}'
    ) AS parsed_result
  FROM {input_fqn}
)
"""

spark.sql(sql)
print(f"Wrote structured events to {output_fqn}.")

In [0]:
# AI Functions occasionally returns null for the strict-schema response — even when
# the underlying prose clearly contains shoplifting evidence. The strict
# responseFormat may be rejecting borderline output, or token budget may be
# truncating the structured response.
#
# Fall back to a much simpler classification call (single-word output, no schema)
# for any row whose assessment ended up NULL on the first pass. This is a
# production pattern: strict structured output as the primary path, simple-output
# fallback for the long tail.

retry_sql = f"""
MERGE INTO {output_fqn} t
USING (
  SELECT
    path,
    upper(trim(
      ai_query(
        '{LLM_ENDPOINT}',
        CONCAT(
          'You are a loss-prevention analyst. Read this CCTV-footage analysis and ',
          'classify it as exactly one of: CONFIRMED, LIKELY, POSSIBLE, NORMAL, ',
          'INSUFFICIENT_EVIDENCE. Reply with just the single word, no explanation. ',
          'Description: ',
          generated_text
        )
      )
    )) AS retry_assessment
  FROM {output_fqn}
  WHERE assessment IS NULL OR assessment = ''
) s
ON t.path = s.path
WHEN MATCHED AND s.retry_assessment IN ('CONFIRMED','LIKELY','POSSIBLE','NORMAL','INSUFFICIENT_EVIDENCE')
THEN UPDATE SET t.assessment = s.retry_assessment
"""

null_count_before = spark.sql(
    f"SELECT count(*) FROM {output_fqn} WHERE assessment IS NULL OR assessment = ''"
).collect()[0][0]

if null_count_before > 0:
    print(f"Retrying {null_count_before} rows with NULL assessment...")
    spark.sql(retry_sql)
    null_count_after = spark.sql(
        f"SELECT count(*) FROM {output_fqn} WHERE assessment IS NULL OR assessment = ''"
    ).collect()[0][0]
    print(f"Retry recovered {null_count_before - null_count_after} of {null_count_before} rows.")
else:
    print("No NULL assessments to retry.")

In [0]:
display(spark.sql(f"""
  SELECT coalesce(assessment, 'NULL') AS assessment, count(*) AS videos, round(avg(confidence), 2) AS avg_confidence
  FROM {output_fqn} GROUP BY 1 ORDER BY videos DESC
"""))

## Demo query

The 5 strongest incidents, with the structured fields an investigator would filter on. The same query runs in the SQL editor against `<catalog>.<schema>.video_events`.

In [0]:
display(spark.sql(f"""
  SELECT
    element_at(split(path, '/'), -1)  AS filename,
    assessment,
    confidence,
    location,
    time_range,
    suspects[0].person_id            AS person,
    suspects[0].item_taken           AS item_taken,
    suspects[0].concealment_method   AS concealment
  FROM {output_fqn}
  WHERE assessment IN ('CONFIRMED', 'LIKELY')
    AND location IS NOT NULL AND location != ''
    AND size(suspects) > 0
  ORDER BY assessment, confidence DESC
  LIMIT 5
"""))